# MuscleMimic mjlab (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/144wHsu_UBVofZqXRTOUWY33ZscziA76R)

Run the MuscleMimic **full-body** task on the **mjlab** backend (Warp + CUDA).
This is the GPU half of `myoMimicFullbody-v0` — parallel `ManagerBasedRlEnv`, not the CPU Gymnasium env.

**Runtime → Change runtime type → T4 GPU** (or better). Warp will not start on a CPU Colab runtime.

1. Install MyoSuite (`ms3`) with `[mjlab,musclemimic]`
2. Download a public KIT walking clip from Hugging Face
3. Register `myoMimicFullbody-v0` via `register_mimic_mjlab_tasks_with_clip`
4. Create a GPU env and step it
5. Short PPO smoke (`MjlabOnPolicyRunner`)
6. Optional rgb preview

`QUICK_MODE` (default) uses few envs / iterations so a T4 can finish. Set `MYOSUITE_FULL_MJLAB=1` for larger GPU training knobs. The optional bimanual dataset is gated — skip it unless you have access.


## 0 — Colab GPU install


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

QUICK_MODE = os.environ.get("MYOSUITE_FULL_MJLAB", "0") != "1"
GIT_URL = os.environ.get("MYOSUITE_GIT_URL", "https://github.com/MyoHub/myosuite.git")
GIT_REF = os.environ.get("MYOSUITE_GIT_REF", "ms3")

if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")


def _find_repo_root() -> Path | None:
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "myosuite" / "__init__.py").is_file():
            return d
    return None


def _has_mjlab_stack() -> bool:
    try:
        import mjlab  # noqa: F401
        import mujoco_warp  # noqa: F401
        import warp  # noqa: F401
        import myosuite.envs.myo.backends.mjlab  # noqa: F401
        return True
    except Exception:
        return False


if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "libegl1", "libgles2", "libosmesa6"],
        check=False,
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "huggingface_hub>=0.20",
            "imageio[ffmpeg]",
            "mediapy",
        ]
    )

repo = _find_repo_root()
if repo is None:
    dest = Path("/content/myosuite") if IN_COLAB else Path.cwd() / "myosuite_src"
    if not dest.exists():
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(dest)]
        )
    if IN_COLAB:
        os.chdir(dest)
    if str(dest) not in sys.path:
        sys.path.insert(0, str(dest))
    repo = dest
else:
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

# Do not force torch>=2.13 from the extra — keep the Colab CUDA torch wheel.
if not _has_mjlab_stack():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo}[musclemimic]"]
    )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "mjlab>=1.4.0",
            "mujoco-warp>=3.7.0",
            "warp-lang",
        ]
    )

print("IN_COLAB:", IN_COLAB, " QUICK_MODE:", QUICK_MODE)
print("cwd:", Path.cwd())

## 1 — Require CUDA

mjlab MuscleMimic uses Warp kernels on GPU. A CPU-only runtime will hang or skip (see the local `11c` notebook). This Colab **stops** if CUDA is missing.


In [ ]:
import platform
import warnings

warnings.filterwarnings("ignore")

import torch
import warp as wp
import mujoco_warp
import mjlab  # noqa: F401
import myosuite.envs.myo.backends.mjlab  # registers mjlab tasks

devices = wp.get_devices()
cuda_ok = torch.cuda.is_available() and any(d.is_cuda for d in devices)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("warp", wp.__version__, "| mujoco_warp", mujoco_warp.__version__)
print("Warp devices:", [str(d) for d in devices])

if not cuda_ok:
    raise RuntimeError(
        "This notebook needs a CUDA GPU. "
        "Runtime → Change runtime type → T4 GPU, then re-run from the top."
    )
if platform.system() == "Darwin":
    raise RuntimeError("mjlab Warp env creation is not supported on macOS here — use Colab GPU.")

_DEVICE = "cuda"
print("Using device:", _DEVICE, torch.cuda.get_device_name(0))

## 2 — Motion clip

Public KIT walking clip (`xpos`, `xquat`, `qpos`, `qvel`, `site_xpos`). Bimanual clips are on a **gated** dataset and are skipped here.


In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download

CLIP_REPO_ID = "amathislab/musclemimic-retargeted"
CLIP_FILENAME = "MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz"

CLIP_PATH = Path(
    hf_hub_download(
        repo_id=CLIP_REPO_ID,
        filename=CLIP_FILENAME,
        repo_type="dataset",
    )
)
if not CLIP_PATH.is_file():
    raise FileNotFoundError(CLIP_PATH)
print("Clip:", CLIP_PATH)

## 3 — Register `myoMimicFullbody-v0`

`load_motion_clip` → `register_mimic_mjlab_tasks_with_clip` (same path as local 11c / `register_mjlab_task`).


In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download
from mjlab.tasks.registry import register_mjlab_task
from myosuite.core.trajectory_io import load_motion_clip
from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
    default_mimic_clip_on_policy_runner_cfg,
    register_mimic_mjlab_tasks_with_clip,
)

CLIP_REPO_ID = "amathislab/musclemimic-retargeted"
CLIP_FILENAME = "MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz"
if "CLIP_PATH" not in globals() or CLIP_PATH is None or not Path(CLIP_PATH).is_file():
    CLIP_PATH = Path(
        hf_hub_download(
            repo_id=CLIP_REPO_ID,
            filename=CLIP_FILENAME,
            repo_type="dataset",
        )
    )
print("Clip:", CLIP_PATH)

clip = load_motion_clip(CLIP_PATH, expected_nq=89, expected_nv=88)
print(f"Clip loaded: T={clip.qpos.shape[0]} frames  nq={clip.qpos.shape[1]}")

register_mimic_mjlab_tasks_with_clip(
    register_mjlab_task=register_mjlab_task,
    rl_cfg_fn=default_mimic_clip_on_policy_runner_cfg,
    clip=clip,
    use_lookahead=True,
)
print("Registered: myoMimicFullbody-v0")

## 4 — GPU env + zero-action rollout

`QUICK_MODE`: 4 parallel envs (raise toward 512–2048 on a larger GPU).


In [ ]:
import os
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download
from mjlab.envs import ManagerBasedRlEnv
from mjlab.tasks.registry import list_tasks, load_env_cfg, register_mjlab_task

if "CLIP_PATH" not in globals() or CLIP_PATH is None or not Path(CLIP_PATH).is_file():
    CLIP_PATH = Path(
        hf_hub_download(
            repo_id="amathislab/musclemimic-retargeted",
            filename="MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz",
            repo_type="dataset",
        )
    )

if "myoMimicFullbody-v0" not in list_tasks():
    from myosuite.core.trajectory_io import load_motion_clip
    from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
        default_mimic_clip_on_policy_runner_cfg,
        register_mimic_mjlab_tasks_with_clip,
    )

    _clip = load_motion_clip(CLIP_PATH, expected_nq=89, expected_nv=88)
    register_mimic_mjlab_tasks_with_clip(
        register_mjlab_task=register_mjlab_task,
        rl_cfg_fn=default_mimic_clip_on_policy_runner_cfg,
        clip=_clip,
        use_lookahead=True,
    )

NUM_ENVS = int(os.environ.get("MJLAB_MIMIC_NUM_ENVS", "4" if QUICK_MODE else "16"))
env_cfg = load_env_cfg("myoMimicFullbody-v0")
env_cfg.scene.num_envs = NUM_ENVS
if os.environ.get("MJLAB_MIMIC_NCONMAX"):
    env_cfg.sim.nconmax = int(os.environ["MJLAB_MIMIC_NCONMAX"])
if os.environ.get("MJLAB_MIMIC_NJMAX"):
    env_cfg.sim.njmax = int(os.environ["MJLAB_MIMIC_NJMAX"])

train_env = ManagerBasedRlEnv(cfg=env_cfg, device=_DEVICE)
obs, _ = train_env.reset()
act_dim = train_env.action_space.shape[-1]
obs_dim = obs["actor"].shape[-1]
print(f"obs_dim={obs_dim}  act_dim={act_dim}  num_envs={NUM_ENVS}")

for _ in range(10):
    actions = torch.zeros(NUM_ENVS, act_dim, device=_DEVICE)
    obs, rew, term, trunc, info = train_env.step(actions)

print("Rollout OK — obs[actor] shape:", tuple(obs["actor"].shape))

## 5 — PPO smoke (`MjlabOnPolicyRunner`)

Uses `default_mimic_clip_on_policy_runner_cfg` so the runner matches registration. `QUICK_MODE` is a few iterations only — not a converged walk.


In [ ]:
import dataclasses
import os

from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper

from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
    default_mimic_clip_on_policy_runner_cfg,
)
from myosuite.envs.myo.backends.mjlab.rsl_rl_logger_episode_patch import (
    install_episode_reward_logging_patch,
)

TRAIN_LOG_DIR = "/tmp/mjlab_mimic_colab"
NUM_ITERS = int(os.environ.get("MJLAB_MIMIC_DEMO_ITERS", "2" if QUICK_MODE else "500"))
STEPS_PER_ENV = int(os.environ.get("MJLAB_MIMIC_STEPS_PER_ENV", "16" if QUICK_MODE else "64"))

wrapped = RslRlVecEnvWrapper(train_env)
runner_cfg = dataclasses.asdict(default_mimic_clip_on_policy_runner_cfg())
runner_cfg["logger"] = os.environ.get("MJLAB_MIMIC_LOGGER", "tensorboard")
runner_cfg["num_steps_per_env"] = STEPS_PER_ENV
runner_cfg["max_iterations"] = NUM_ITERS
runner_cfg["save_interval"] = min(100, max(1, NUM_ITERS))

install_episode_reward_logging_patch()
runner = MjlabOnPolicyRunner(
    env=wrapped,
    train_cfg=runner_cfg,
    log_dir=TRAIN_LOG_DIR,
    device=_DEVICE,
)
runner.learn(num_learning_iterations=NUM_ITERS, init_at_random_ep_len=True)
print("Training complete — checkpoints in", TRAIN_LOG_DIR)

## 6 — Render a short preview

Loads the latest `model_*.pt` if present; otherwise zeros. Needs `render_mode="rgb_array"`.


In [ ]:
import glob
import os
import re
from pathlib import Path

import mediapy
import torch
from huggingface_hub import hf_hub_download
from IPython.display import Video, display
from mjlab.envs import ManagerBasedRlEnv
from mjlab.tasks.registry import list_tasks, load_env_cfg, register_mjlab_task
from mjlab.viewer import ViewerConfig

from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
    default_mimic_clip_on_policy_runner_cfg,
    register_mimic_mjlab_tasks_with_clip,
)

N_FRAMES = int(os.environ.get("MJLAB_MIMIC_RENDER_FRAMES", "60" if QUICK_MODE else "300"))
FPS = 30
VIDEO_PATH = "/tmp/mjlab_policy_rollout.mp4"


def _ckpt_iter(path: str) -> int:
    m = re.search(r"model_(\d+)\.pt$", path.replace("\\", "/"))
    return int(m.group(1)) if m else -1


ckpts = sorted(
    glob.glob(f"{TRAIN_LOG_DIR}/**/model_*.pt", recursive=True),
    key=_ckpt_iter,
)
policy_fn = None
if ckpts:
    latest = ckpts[-1]
    print("Loading checkpoint:", latest)
    with torch.inference_mode():
        runner.load(latest, map_location=_DEVICE)
    policy_fn = runner.get_inference_policy(device=_DEVICE)
else:
    print("No checkpoint — zero-action rollout.")

if "CLIP_PATH" not in globals() or CLIP_PATH is None or not Path(CLIP_PATH).is_file():
    CLIP_PATH = Path(
        hf_hub_download(
            repo_id="amathislab/musclemimic-retargeted",
            filename="MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz",
            repo_type="dataset",
        )
    )

if "myoMimicFullbody-v0" not in list_tasks():
    from myosuite.core.trajectory_io import load_motion_clip

    _clip = load_motion_clip(CLIP_PATH, expected_nq=89, expected_nv=88)
    register_mimic_mjlab_tasks_with_clip(
        register_mjlab_task=register_mjlab_task,
        rl_cfg_fn=default_mimic_clip_on_policy_runner_cfg,
        clip=_clip,
        use_lookahead=True,
    )

render_cfg = load_env_cfg("myoMimicFullbody-v0", play=True)
render_cfg.scene.num_envs = 1
render_cfg.viewer = ViewerConfig(width=640, height=360, distance=3.5, elevation=-20.0)

render_env = ManagerBasedRlEnv(cfg=render_cfg, device=_DEVICE, render_mode="rgb_array")
obs, _ = render_env.reset()
act_dim_r = render_env.action_space.shape[-1]

frames = []
for _ in range(N_FRAMES):
    if policy_fn is not None:
        with torch.no_grad():
            actions = policy_fn(obs)
    else:
        actions = torch.zeros(1, act_dim_r, device=_DEVICE)
    obs, _, _, _, _ = render_env.step(actions)
    frame = render_env.render()
    if frame is not None:
        frames.append(frame)

render_env.close()
if not frames:
    raise RuntimeError("No frames captured; check render_mode and viewer config.")
print(f"Captured {len(frames)} frames  ({frames[0].shape} each)")
mediapy.write_video(VIDEO_PATH, frames, fps=FPS)
display(Video(VIDEO_PATH, embed=True, width=640, height=360))

## 7 — Architecture

```
motion clip (.npz)
  └── load_motion_clip()
        └── register_mimic_mjlab_tasks_with_clip()
              └── ManagerBasedRlEnvCfg  (same env_id as the CPU Gymnasium task)
                    └── ManagerBasedRlEnv  (Warp / CUDA)
                          └── RslRlVecEnvWrapper
                                └── MjlabOnPolicyRunner
```

Local / CI companion (skips Warp on macOS / CPU): `11c_MuscleMimic_Fullbody_mjlab.ipynb`.
CPU directional student (no Warp): `11d_MuscleMimic_Fullbody_directional_locomotion.ipynb`.
